# Generate the interactive GIF pictures of the recursive splitting of the BPT/AA structure of an input image

In [ ]:
import numpy as np
import json
import sys, os, importlib, math
import matplotlib.pyplot as plt
import cv2
from tqdm.auto import tqdm
import io, imageio.v2 as imageio

# Keep text rendering notebook-portable. usetex=True often fails on machines
# without a full LaTeX installation.
plt.rcParams['text.usetex'] = False

import shap_bpt as shap_bpt
print('shap_bpt version:', shap_bpt.__version__)


In [ ]:
import os
from pathlib import Path
import yaml

def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start).resolve()
    for path in (start, *start.parents):
        if (path / 'pyproject.toml').exists() and (path / 'shap_bpt').is_dir():
            return path
    raise FileNotFoundError('Could not find the project root from the current working directory.')

original_working_dir = Path.cwd()
project_root = find_project_root(original_working_dir)
os.chdir(project_root)
print(f'Project root: {project_root}')

config_file = "MSCOCO_mac"
# config_file = "MSCOCO_xn2"

try:
    with open(project_root / f"examples/configs/{config_file}.yaml", "r") as f:
        config = yaml.safe_load(f)
finally:
    os.chdir(original_working_dir)
    print(f'Restored working directory: {original_working_dir}')

dataset_root = config["data"]["dataset_root"]
print(f"Dataset root: {dataset_root}")

In [ ]:
# config['data']['partition_size'] = 100
config['data']['partition_size'] = 24


In [ ]:
path_partitions = project_root / f"examples/partitions_{config['data']['partition_size']}"
path_partitions

In [ ]:
## Get available precomputed SAM partitions
# fetch available unique image_ids from the partitions directory
image_ids = os.listdir(path_partitions)
image_ids = [f.split('.')[0].split('_')[0] for f in image_ids if '_refined.npy' in f]
print(f'computed image_ids: {len(image_ids)}')

already_computed = ['000000049091',
 '000000000632',
 '000000186929',
 '000000002299',
 '000000225757',
 '000000171382']

image_ids = [img_id for img_id in image_ids if img_id not in already_computed]
print(f'filtered image_ids: {len(image_ids)}')
image_ids
image_dir = config["data"]["image_dir"] # Update for your image directory

In [ ]:
image_id = image_ids[0]  # Example image ID
image_path = os.path.join(image_dir, f'{image_id}.jpg')

In [ ]:
image_to_explain = cv2.imread(image_path, cv2.IMREAD_COLOR)[:,:,::-1].astype(np.uint8)
print(image_to_explain.shape)

In [ ]:
%%time
bptree = shap_bpt.build_bpt_from_image(image_to_explain)

In [ ]:
# Utilities to visualize the real BPT regions built from the image.
# These are not rectangular approximations: each frontier node is drawn using
# the exact pixels stored in the BPT.

def bpt_node_interval(bptree, node_index):
    if node_index < bptree.U:
        start = int(bptree.leaf_idx[node_index])
        end = start + 1
    else:
        offset = node_index - bptree.U
        start = int(bptree.cl_start[offset])
        end = int(bptree.cl_end[offset])
    return start, end


def bpt_node_pixels(bptree, node_index):
    start, end = bpt_node_interval(bptree, node_index)
    return bptree.pixels[start:end].astype(np.int64)


def bpt_node_area(bptree, node_index):
    start, end = bpt_node_interval(bptree, node_index)
    return int(end - start)


def bpt_children(bptree, node_index):
    if node_index < bptree.U:
        return None
    offset = node_index - bptree.U
    return int(bptree.cl_left[offset]), int(bptree.cl_right[offset])


def split_bpt_frontier_once(bptree, frontier, split_all=True):
    if split_all:
        next_frontier = []
        changed = False
        for node_index in frontier:
            children = bpt_children(bptree, node_index)
            if children is None:
                next_frontier.append(node_index)
            else:
                next_frontier.extend(children)
                changed = True
        return next_frontier, changed

    split_candidates = [idx for idx in frontier if bpt_children(bptree, idx) is not None]
    if not split_candidates:
        return frontier, False
    target = max(split_candidates, key=lambda idx: bpt_node_area(bptree, idx))
    next_frontier = []
    for node_index in frontier:
        if node_index == target:
            next_frontier.extend(bpt_children(bptree, node_index))
        else:
            next_frontier.append(node_index)
    return next_frontier, True


def bpt_frontiers_by_depth(bptree, max_depth=5):
    frontiers = []
    frontier = [bptree.N - 1]
    for depth in range(max_depth + 1):
        frontiers.append(list(frontier))
        frontier, changed = split_bpt_frontier_once(bptree, frontier, split_all=True)
        if not changed:
            break
    return frontiers


def bpt_frontier_label_image(bptree, image_shape, frontier):
    height, width = image_shape[:2]
    labels = np.zeros((height, width), dtype=np.int32)
    flat = labels.ravel()
    for label, node_index in enumerate(frontier, start=1):
        flat[bpt_node_pixels(bptree, node_index)] = label
    return labels


def label_boundaries(labels):
    boundaries = np.zeros(labels.shape, dtype=bool)
    boundaries[:-1, :] |= labels[:-1, :] != labels[1:, :]
    boundaries[1:, :] |= labels[:-1, :] != labels[1:, :]
    boundaries[:, :-1] |= labels[:, :-1] != labels[:, 1:]
    boundaries[:, 1:] |= labels[:, :-1] != labels[:, 1:]
    return boundaries


def colorize_bpt_frontier(bptree, image, frontier):
    # Fill each actual BPT region with its mean RGB color. This gives a compact
    # stock BPT segmentation view while preserving the image-derived regions.
    flat_image = image.reshape((-1, image.shape[-1])).astype(np.float32)
    colored = np.zeros_like(flat_image, dtype=np.float32)
    for node_index in frontier:
        pixels = bpt_node_pixels(bptree, node_index)
        if len(pixels) == 0:
            continue
        colored[pixels] = flat_image[pixels].mean(axis=0)
    colored = colored.reshape(image.shape)
    return np.clip(colored / 255.0, 0, 1)


def draw_bpt_frontier(bptree, image, frontier, ax=None, title=None,
                      image_alpha=0.55, boundary_color=(1.0, 0.0, 0.15),
                      show_mean_regions=True):
    if ax is None:
        fig, ax = plt.subplots(figsize=(6, 6))
    else:
        fig = ax.figure

    base = image.astype(np.float32) / 255.0 if image.max() > 1 else image.astype(np.float32)
    labels = bpt_frontier_label_image(bptree, image.shape, frontier)
    boundaries = label_boundaries(labels)

    if show_mean_regions:
        display = colorize_bpt_frontier(bptree, image, frontier)
        display = np.clip(0.18 + 0.90 * display, 0, 1)
    else:
        display = base.copy()

    display = image_alpha * display + (1.0 - image_alpha) * base
    display[boundaries] = boundary_color

    ax.imshow(display)
    ax.set_xticks([])
    ax.set_yticks([])
    if title is not None:
        ax.set_title(title)
    return fig, ax, labels


def make_bpt_region_sequence(bptree, image, frames_count=10, split_all=True,
                             save_gif=True, gif_path='sequence_bpt_regions.gif',
                             save_last_png=True, png_path='bpt_regions_last.png'):
    frontier = [bptree.N - 1]
    frames = []
    last_fig = None

    for step in range(frames_count):
        fig, ax = plt.subplots(figsize=(5, 5))
        draw_bpt_frontier(
            bptree,
            image,
            frontier,
            ax=ax,
            title=f'BPT regions - depth {step} ({len(frontier)} regions)',
            image_alpha=0.75,
            show_mean_regions=True,
        )
        fig.tight_layout()

        if save_gif:
            buf = io.BytesIO()
            fig.savefig(buf, format='png', dpi=120)
            buf.seek(0)
            frames.append(imageio.imread(buf))
            buf.close()

        if step == frames_count - 1 and save_last_png:
            fig.savefig(png_path, dpi=200, bbox_inches='tight')
            last_fig = fig
        else:
            plt.close(fig)

        frontier, changed = split_bpt_frontier_once(bptree, frontier, split_all=split_all)
        if not changed:
            break

    if save_gif and frames:
        durations = [900] + [500] * max(0, len(frames) - 2) + ([1300] if len(frames) > 1 else [])
        imageio.mimsave(gif_path, frames, duration=durations[:len(frames)], loop=0)
        print(f'Saved GIF: {gif_path}')
    if save_last_png:
        print(f'Saved last-frame PNG: {png_path}')
    return frontier, last_fig


def plot_bpt_depth_grid(bptree, image, max_depth=5, ncols=3,
                        figsize_per_panel=(4, 4), save_path='bpt_depth_grid.png'):
    frontiers = bpt_frontiers_by_depth(bptree, max_depth=max_depth)
    n_panels = len(frontiers)
    ncols = min(ncols, n_panels)
    nrows = int(np.ceil(n_panels / ncols))

    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=(figsize_per_panel[0] * ncols, figsize_per_panel[1] * nrows),
        squeeze=False,
    )
    axes = axes.ravel()

    for depth, frontier in enumerate(frontiers):
        draw_bpt_frontier(
            bptree,
            image,
            frontier,
            ax=axes[depth],
            title=f'Depth {depth}: {len(frontier)} regions',
            image_alpha=0.75,
            show_mean_regions=True,
        )

    for ax in axes[n_panels:]:
        ax.axis('off')

    fig.tight_layout()
    if save_path is not None:
        fig.savefig(save_path, dpi=200, bbox_inches='tight')
        print(f'Saved depth grid: {save_path}')
    return fig, axes, frontiers


def plot_bpt_tree_structure(bptree, max_depth=4, save_path='bpt_tree_structure.png'):
    levels = [[bptree.N - 1]]
    for _ in range(max_depth):
        next_level = []
        for node_index in levels[-1]:
            children = bpt_children(bptree, node_index)
            if children is not None:
                next_level.extend(children)
        if not next_level:
            break
        levels.append(next_level)

    fig, ax = plt.subplots(figsize=(max(7, 1.2 * max(len(level) for level in levels)), 1.4 * len(levels)))
    ax.axis('off')

    positions = {}
    for depth, level in enumerate(levels):
        xs = np.linspace(0, 1, len(level) + 2)[1:-1]
        y = 1 - depth / max(1, len(levels) - 1)
        for x, node_index in zip(xs, level):
            positions[node_index] = (x, y)
            area = bpt_node_area(bptree, node_index)
            ax.text(
                x,
                y,
                f'{node_index}\n{area}px',
                ha='center',
                va='center',
                fontsize=8,
                bbox=dict(boxstyle='round,pad=0.25', fc='white', ec='black', lw=0.8),
            )

    for level in levels[:-1]:
        for node_index in level:
            children = bpt_children(bptree, node_index)
            if children is None:
                continue
            x0, y0 = positions[node_index]
            for child in children:
                if child not in positions:
                    continue
                x1, y1 = positions[child]
                ax.plot([x0, x1], [y0 - 0.025, y1 + 0.025], color='0.35', lw=1.0, zorder=0)

    ax.set_title(f'BPT tree structure up to depth {len(levels) - 1}')
    if save_path is not None:
        fig.savefig(save_path, dpi=200, bbox_inches='tight')
        print(f'Saved tree structure: {save_path}')
    return fig, ax, levels


def cap_partition_labels_local(partitions, max_labels=63):
    from scipy import ndimage as ndi

    partitions = np.asarray(partitions).astype(np.int64)
    labels, counts = np.unique(partitions, return_counts=True)

    if len(labels) <= max_labels:
        keep_labels = labels
    else:
        keep_labels = labels[np.argsort(counts)[-max_labels:]]

    keep_mask = np.isin(partitions, keep_labels)
    if not np.all(keep_mask):
        _, nearest_idx = ndi.distance_transform_edt(~keep_mask, return_indices=True)
        partitions = partitions.copy()
        partitions[~keep_mask] = partitions[tuple(idx[~keep_mask] for idx in nearest_idx)]

    unique_ids = np.unique(partitions)
    remap = {old: new for new, old in enumerate(unique_ids)}
    return np.vectorize(remap.get)(partitions).astype(np.uint8)


def sanitize_partitions_local(partitions, image_shape, max_labels=63):
    p = np.asarray(partitions).astype(np.int64, copy=True)
    if p.shape != image_shape[:2]:
        raise ValueError(f'Expected partitions shape {image_shape[:2]}, got {p.shape}')

    p[p < 0] = 0
    labels = [label for label in np.unique(p) if label > 0]
    labels = sorted(labels, key=lambda label: np.sum(p == label), reverse=True)[:max_labels]

    out = np.zeros_like(p, dtype=np.int64)
    for new_label, old_label in enumerate(labels, start=1):
        out[p == old_label] = new_label
    return out.astype(np.uint8)


def prepare_partitions(partitions, image_shape, max_labels=63):
    capped = cap_partition_labels_local(partitions, max_labels=max_labels)
    return sanitize_partitions_local(capped, image_shape, max_labels=max_labels)


def aa_frontier_by_depth(image_shape, depth=5):
    height, width = image_shape[:2]
    frontier = [(0, width, 0, height)]
    for _ in range(depth):
        next_frontier = []
        for xmin, xmax, ymin, ymax in frontier:
            size_x = xmax - xmin
            size_y = ymax - ymin
            if size_x <= 1 and size_y <= 1:
                next_frontier.append((xmin, xmax, ymin, ymax))
            elif size_x > size_y and size_x > 1:
                xmid = xmin + size_x // 2
                next_frontier.extend([(xmin, xmid, ymin, ymax), (xmid, xmax, ymin, ymax)])
            else:
                ymid = ymin + size_y // 2
                next_frontier.extend([(xmin, xmax, ymin, ymid), (xmin, xmax, ymid, ymax)])
        frontier = next_frontier
    return frontier


def aa_label_image(image_shape, depth=5):
    labels = np.zeros(image_shape[:2], dtype=np.int32)
    for label, (xmin, xmax, ymin, ymax) in enumerate(aa_frontier_by_depth(image_shape, depth=depth), start=1):
        labels[ymin:ymax, xmin:xmax] = label
    return labels


def bpt_label_image_at_depth(bptree, image_shape, depth=5):
    frontiers = bpt_frontiers_by_depth(bptree, max_depth=depth)
    return bpt_frontier_label_image(bptree, image_shape, frontiers[-1])


def mean_region_overlay_from_labels(image, labels, image_alpha=0.55, boundary_color=(1.0, 0.0, 0.15)):
    base = image.astype(np.float32) / 255.0 if image.max() > 1 else image.astype(np.float32)
    flat_image = image.reshape((-1, image.shape[-1])).astype(np.float32)
    colored = np.zeros_like(flat_image, dtype=np.float32)
    flat_labels = labels.ravel()

    for label in np.unique(flat_labels):
        mask = flat_labels == label
        if not np.any(mask):
            continue
        colored[mask] = flat_image[mask].mean(axis=0)

    colored = np.clip(colored.reshape(image.shape) / 255.0, 0, 1)
    display = image_alpha * np.clip(0.18 + 0.90 * colored, 0, 1) + (1.0 - image_alpha) * base
    display[label_boundaries(labels)] = boundary_color
    return display


def draw_label_regions(image, labels, ax=None, title=None, image_alpha=0.55):
    if ax is None:
        fig, ax = plt.subplots(figsize=(5, 5))
    else:
        fig = ax.figure
    ax.imshow(mean_region_overlay_from_labels(image, labels, image_alpha=image_alpha))
    ax.set_xticks([])
    ax.set_yticks([])
    if title is not None:
        ax.set_title(title)
    return fig, ax


def compare_partition_methods(image, bptree, partitions_sorted, partitions_refined,
                              bptree_sam_sorted, bptree_sam_refined,
                              depth=5, save_path='bpt_sam_partition_comparison.png'):
    panels = [
        ('AA', aa_label_image(image.shape, depth=depth)),
        ('BPT', bpt_label_image_at_depth(bptree, image.shape, depth=depth)),
        ('SAM (Sorted)', partitions_sorted),
        # ('SAM (Refined)', partitions_refined),
        ('SAM+BPT (Sorted)', bpt_label_image_at_depth(bptree_sam_sorted, image.shape, depth=depth)),
        ('SAM+BPT (Refined)', bpt_label_image_at_depth(bptree_sam_refined, image.shape, depth=depth)),
    ]

    fig, axes = plt.subplots(2, 3, figsize=(13, 8), squeeze=False)
    axes = axes.ravel()
    for ax, (title, labels) in zip(axes, panels):
        draw_label_regions(image, labels, ax=ax, title=f'{title}\n{len(np.unique(labels))} regions')

    fig.suptitle(f'Partition Comparison at Depth {depth}', fontsize=16)
    fig.tight_layout()
    if save_path is not None:
        fig.savefig(save_path, dpi=200, bbox_inches='tight')
        print(f'Saved comparison: {save_path}')
    return fig, axes, panels



def comparison_panels_at_depth(image, bptree, partitions_sorted, partitions_refined,
                               bptree_sam_sorted, bptree_sam_refined, depth=5):
    return [
        ('AA', aa_label_image(image.shape, depth=depth)),
        ('BPT', bpt_label_image_at_depth(bptree, image.shape, depth=depth)),
        ('SAM (Sorted)', partitions_sorted),
        # ('SAM (Refined)', partitions_refined),
        ('SAM+BPT (Sorted)', bpt_label_image_at_depth(bptree_sam_sorted, image.shape, depth=depth)),
        ('SAM+BPT (Refined)', bpt_label_image_at_depth(bptree_sam_refined, image.shape, depth=depth)),
    ]


def plot_partition_comparison_depth(image, bptree, partitions_sorted, partitions_refined,
                                    bptree_sam_sorted, bptree_sam_refined,
                                    depth=5, save_path=None):
    panels = comparison_panels_at_depth(
        image,
        bptree,
        partitions_sorted,
        partitions_refined,
        bptree_sam_sorted,
        bptree_sam_refined,
        depth=depth,
    )

    fig, axes = plt.subplots(2, 3, figsize=(13, 8), squeeze=False)
    axes = axes.ravel()
    for ax, (title, labels) in zip(axes, panels):
        draw_label_regions(image, labels, ax=ax, title=f'{title}\n{len(np.unique(labels))} regions')

    fig.suptitle(f'Partition Comparison at Depth {depth}', fontsize=16)
    fig.tight_layout()
    if save_path is not None:
        fig.savefig(save_path, dpi=200, bbox_inches='tight')
        print(f'Saved comparison frame: {save_path}')
    return fig, axes, panels


def make_partition_comparison_gif(image, bptree, partitions_sorted, partitions_refined,
                                  bptree_sam_sorted, bptree_sam_refined,
                                  max_depth=7,
                                  gif_path='bpt_sam_partition_comparison_depths.gif',
                                  png_path=None,
                                  duration_ms=700,
                                  dpi=150):
    frames = []
    last_fig = None
    for depth in range(max_depth + 1):
        fig, axes, panels = plot_partition_comparison_depth(
            image,
            bptree,
            partitions_sorted,
            partitions_refined,
            bptree_sam_sorted,
            bptree_sam_refined,
            depth=depth,
            save_path=None,
        )
        buffer = io.BytesIO()
        fig.savefig(buffer, format='png', dpi=dpi, bbox_inches='tight')
        buffer.seek(0)
        frames.append(imageio.imread(buffer))
        if depth < max_depth:
            plt.close(fig)
        else:
            last_fig = fig

    imageio.mimsave(gif_path, frames, duration=duration_ms / 1000.0, loop=0)
    print(f'Saved GIF: {gif_path}')
    if png_path is not None and last_fig is not None:
        last_fig.savefig(png_path, dpi=200, bbox_inches='tight')
        print(f'Saved final frame: {png_path}')
    return frames



def method_labels_at_depth(image, method, depth=4):
    # method can be ('Name', 'aa'), ('Name', bptree), ('Name', label_image),
    # a dict with name/kind/bpt/labels, or a callable(image, depth) -> labels.
    if isinstance(method, dict):
        name = method.get('name', method.get('title', 'Method'))
        if method.get('kind') == 'aa':
            labels = aa_label_image(image.shape, depth=depth)
        elif 'bpt' in method:
            labels = bpt_label_image_at_depth(method['bpt'], image.shape, depth=depth)
        elif 'labels' in method:
            labels = np.asarray(method['labels'])
        elif 'fn' in method:
            labels = np.asarray(method['fn'](image, depth))
        else:
            raise ValueError(f'Unsupported method dictionary: {method}')
        return name, labels

    name, source = method
    if isinstance(source, str) and source.lower() == 'aa':
        return name, aa_label_image(image.shape, depth=depth)
    if callable(source):
        return name, np.asarray(source(image, depth))
    if hasattr(source, 'pixels') and hasattr(source, 'N'):
        return name, bpt_label_image_at_depth(source, image.shape, depth=depth)
    return name, np.asarray(source)


def plot_methods_depth_grid(image, methods, depth=4, save_path=None, figsize=None, dpi=200):
    depth_values = list(range(1, depth + 1))
    n_rows = len(depth_values)
    n_cols = len(methods)
    if figsize is None:
        figsize = (3.2 * n_cols, 3.0 * n_rows)

    fig, axes = plt.subplots(n_rows, n_cols, figsize=figsize, squeeze=False)
    panels_by_depth = []
    for row, current_depth in enumerate(depth_values):
        row_panels = []
        for col, method in enumerate(methods):
            ax = axes[row, col]
            name, labels = method_labels_at_depth(image, method, depth=current_depth)
            row_panels.append((name, labels))
            title = f'{name}\nDepth {current_depth}, {len(np.unique(labels))} regions'
            draw_label_regions(image, labels, ax=ax, title=title)
        axes[row, 0].set_ylabel(f'Depth {current_depth}', fontsize=13)
        panels_by_depth.append(row_panels)

    fig.suptitle(f'Method Comparison Across Depths 1-{depth}', fontsize=16)
    fig.tight_layout()
    if save_path is not None:
        fig.savefig(save_path, dpi=dpi, bbox_inches='tight')
        print(f'Saved depth grid: {save_path}')
    return fig, axes, panels_by_depth


def plot_methods_single_depth(image, methods, depth=4, save_path=None, figsize=None, dpi=200):
    if figsize is None:
        figsize = (3.2 * len(methods), 3.2)
    fig, axes = plt.subplots(1, len(methods), figsize=figsize, squeeze=False)
    axes = axes.ravel()
    panels = []
    for ax, method in zip(axes, methods):
        name, labels = method_labels_at_depth(image, method, depth=depth)
        panels.append((name, labels))
        draw_label_regions(image, labels, ax=ax, title=f'{name}\n{len(np.unique(labels))} regions')
    fig.suptitle(f'Method Comparison at Depth {depth}', fontsize=16)
    fig.tight_layout()
    if save_path is not None:
        fig.savefig(save_path, dpi=dpi, bbox_inches='tight')
        print(f'Saved depth frame: {save_path}')
    return fig, axes, panels


def make_methods_depth_gif(image, methods, depth=4, gif_path='methods_depth_comparison.gif',
                           png_path=None, duration_ms=700, dpi=150):
    frames = []
    last_fig = None
    for current_depth in range(1, depth + 1):
        fig, axes, panels = plot_methods_single_depth(
            image,
            methods,
            depth=current_depth,
            save_path=None,
            dpi=dpi,
        )
        buffer = io.BytesIO()
        fig.savefig(buffer, format='png', dpi=dpi, bbox_inches='tight')
        buffer.seek(0)
        frames.append(imageio.imread(buffer))
        if current_depth < depth:
            plt.close(fig)
        else:
            last_fig = fig

    imageio.mimsave(gif_path, frames, duration=duration_ms / 1000.0, loop=0)
    print(f'Saved GIF: {gif_path}')
    if png_path is not None and last_fig is not None:
        last_fig.savefig(png_path, dpi=200, bbox_inches='tight')
        print(f'Saved final frame: {png_path}')
    return frames


def generate_methods_depth_comparison(image, methods, depth=4,
                                      grid_path='methods_depth_grid.png',
                                      gif_path='methods_depth_comparison.gif',
                                      final_frame_path=None,
                                      duration_ms=700):
    fig, axes, panels_by_depth = plot_methods_depth_grid(
        image,
        methods,
        depth=depth,
        save_path=grid_path,
    )
    frames = make_methods_depth_gif(
        image,
        methods,
        depth=depth,
        gif_path=gif_path,
        png_path=final_frame_path,
        duration_ms=duration_ms,
    )
    return fig, axes, panels_by_depth, frames


In [ ]:
# Visualize the real BPT regions built from the image.
# split_all=True shows hierarchy depth levels. Use split_all=False to split one
# largest BPT region at a time for a slower animation.
frontier, last_fig = make_bpt_region_sequence(
    bptree,
    image_to_explain,
    frames_count=11,
    split_all=True,
    save_gif=True,
    gif_path='/Users/rashid/Nextcloud/RashidPHD/Codes/XAI/ShapBPT-SAM/shap_bpt_sam/examples/notebooks/splits/sequence_bpt_regions.gif',
    save_last_png=True,
    png_path='/Users/rashid/Nextcloud/RashidPHD/Codes/XAI/ShapBPT-SAM/shap_bpt_sam/examples/notebooks/splits/bpt_regions_last.png',
)
plt.show()


In [ ]:
# Load SAM sorted/refined masks, build BPTs using those same partitions,
# and compare AA, BPT, SAM, and SAM+BPT regions.
masks_sorted = np.load(f"{path_partitions}/{image_id}_sorted.npy").astype(np.uint16)
masks_refined = np.load(f"{path_partitions}/{image_id}_refined.npy").astype(np.uint16)

partitions_sorted = prepare_partitions(masks_sorted, image_to_explain.shape, max_labels=63)
partitions_refined = prepare_partitions(masks_refined, image_to_explain.shape, max_labels=63)

bptree_sam_sorted = shap_bpt.build_bpt_from_image(
    image_to_explain,
    prebuilt_partitions=partitions_sorted,
)
bptree_sam_refined = shap_bpt.build_bpt_from_image(
    image_to_explain,
    prebuilt_partitions=partitions_refined,
)

comparison_depth = 5
fig, axes, comparison_panels = compare_partition_methods(
    image_to_explain,
    bptree,
    partitions_sorted,
    partitions_refined,
    bptree_sam_sorted,
    bptree_sam_refined,
    depth=comparison_depth,
    save_path=f'/Users/rashid/Nextcloud/RashidPHD/Codes/XAI/ShapBPT-SAM/shap_bpt_sam/examples/notebooks/splits/bpt_sam_partition_comparison_depth{comparison_depth}_{image_id}.png',
)
plt.show()


In [ ]:
# Animate the comparison across BPT depths.
# AA, BPT, SAM+BPT(Sorted), and SAM+BPT(Refined) evolve with depth;
# SAM(Sorted) and SAM(Refined) stay fixed for reference.
max_animation_depth = 7
comparison_frames = make_partition_comparison_gif(
    image_to_explain,
    bptree,
    partitions_sorted,
    partitions_refined,
    bptree_sam_sorted,
    bptree_sam_refined,
    max_depth=max_animation_depth,
    gif_path=f'/Users/rashid/Nextcloud/RashidPHD/Codes/XAI/ShapBPT-SAM/shap_bpt_sam/examples/notebooks/splits/bpt_sam_partition_comparison_depths_{image_id}.gif',
    png_path=f'/Users/rashid/Nextcloud/RashidPHD/Codes/XAI/ShapBPT-SAM/shap_bpt_sam/examples/notebooks/splits/bpt_sam_partition_comparison_depth{max_animation_depth}_{image_id}.png',
    duration_ms=700,
)


In [ ]:
# Simple 4-depth x 6-method comparison using a method list.
# BPT-like entries are animated by depth; label-image entries stay fixed.
comparison_methods = [
    ('AA', 'aa'),
    ('BPT', bptree),
    ('SAM (Sorted)', partitions_sorted),
    # ('SAM (Refined)', partitions_refined),
    ('SAM+BPT (Sorted)', bptree_sam_sorted),
    ('SAM+BPT (Refined)', bptree_sam_refined),
]

comparison_grid_fig, comparison_grid_axes, comparison_grid_panels, comparison_grid_frames = generate_methods_depth_comparison(
    image_to_explain,
    comparison_methods,
    depth=4,
    # grid_path=f'/Users/rashid/Nextcloud/RashidPHD/Codes/XAI/ShapBPT-SAM/shap_bpt_sam/examples/notebooks/splits/bpt_sam_methods_depth_grid_{image_id}.png',
    gif_path=f' /Users/rashid/Nextcloud/RashidPHD/Codes/XAI/ShapBPT-SAM/shap_bpt_sam/examples/notebooks/splits/bpt_sam_methods_depth_gif_{image_id}.gif',
    # final_frame_path=f'/Users/rashid/Nextcloud/RashidPHD/Codes/XAI/ShapBPT-SAM/shap_bpt_sam/examples/notebooks/splits/bpt_sam_methods_depth4_{image_id}.png',
    duration_ms=700,
)
